# Notebook 4: Preprocessing Clarifications

In this notebook, we focus on key preprocessing steps for hyperscanning analysis:

1. **Union vs. Intersection of Epochs:**  
   Understand the impact of choosing to keep either all valid epochs (union) or only the epochs common to both participants (intersection).

2. **Artifact Rejection:**  
   - **EEG:** Use ICA to remove artifacts (e.g., eye blinks, muscle activity).  
   - **fNIRS:** Apply short-separation regression to reduce extracerebral (scalp) contamination.

3. **Arbitrary Methodological Choices:**  
   Discuss how filtering, baseline correction, and reference choices can influence results and why it's critical to document these decisions.

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# Hypothetical modules from HyPyP for preprocessing (assumed to be available)
import hypyp.prep as prep  # Contains functions for ICA rejection, short-separation regression, etc.
from hypyp.fnirs_tools import load_fnirs

print("Libraries imported successfully.")

## 1. Union vs. Intersection of Epochs

When preprocessing data from two participants, you may encounter situations where one participant has more valid epochs than the other.  
- **Union of Epochs:**  
  Keep all valid epochs from each participant. However, the epoch counts might differ, and care must be taken when computing inter-brain synchrony.
  
- **Intersection of Epochs:**  
  Only keep epochs that are valid for both participants. This ensures that the data is directly comparable but may reduce the overall amount of data.

Below is an example using MNE to achieve the intersection of epochs.

In [ ]:
# Load example epochs (adjust file paths accordingly)
epo1 = mne.read_epochs("./data/participant1-epo.fif", preload=True)
epo2 = mne.read_epochs("./data/participant2-epo.fif", preload=True)

# Using MNE's equalize_epoch_counts ensures we only use epochs that are valid for both participants.
mne.epochs.equalize_epoch_counts([epo1, epo2])

print("Epochs after intersection:")
print("Participant 1:", epo1)
print("Participant 2:", epo2)

In [ ]:
# Suppose the raw montage has the following channels (example):
all_channels = epo1.ch_names  # assume both participants share the same channel list

# Mark bad channels:
epo1.info['bads'] = ['Fp1']  # Participant 1 has FP1 as bad
epo2.info['bads'] = ['Cz']   # Participant 2 has CZ as bad

# Compute the "good" channels for each participant (union approach)
good_channels_p1 = [ch for ch in all_channels if ch not in epo1.info['bads']]
good_channels_p2 = [ch for ch in all_channels if ch not in epo2.info['bads']]

print("Union Approach:")
print("Participant 1 good channels:", good_channels_p1)
print("Participant 2 good channels:", good_channels_p2)

# Intersection approach: only keep channels that are good in both participants
common_good_channels = list(set(good_channels_p1).intersection(set(good_channels_p2)))
print("\nIntersection Approach:")
print("Common good channels:", sorted(common_good_channels))

# To apply the intersection approach, restrict each epochs object to the common good channels:
epo1_intersect = epo1.copy().pick(common_good_channels)
epo2_intersect = epo2.copy().pick(common_good_channels)

print("\nAfter applying intersection:")
print("Participant 1 channels:", epo1_intersect.ch_names)
print("Participant 2 channels:", epo2_intersect.ch_names)

## 2. Artifact Rejection: ICA for EEG and Short-Separation Regression for fNIRS

### 2.1 ICA for EEG
ICA can help remove artifacts such as eye blinks and muscle activity.

In [ ]:
# Compute ICA for each participant with 15 components
icas = prep.ICA_fit([
    epo1, epo2
],
    n_components=15,
    method='infomax',
    fit_params=dict(extended=True),
    random_state=42
)

# Select the relevant independent components for artefact rejection
cleaned_epochs_ICA = prep.ICA_choice_comp(icas, [epo1, epo2])
print('ICA correction completed.')

# Apply local AutoReject on the ICA-cleaned epochs
cleaned_epochs_AR, dic_AR = prep.AR_local(
    cleaned_epochs_ICA,
    strategy="union",
    threshold=50.0,
    verbose=True
)
print('AutoReject completed.')

# Assign cleaned epochs to individual participant variables
epo1_clean = cleaned_epochs_AR[0]
epo2_clean = cleaned_epochs_AR[1]
print('Preprocessed epochs for both participants are ready.')

# Update dyad with cleaned data for subsequent analysis
dyad_clean = [epo1_clean.get_data(copy=True), epo2_clean.get_data(copy=True)]

### 2.2 Short-Separation Regression for fNIRS
For fNIRS data, short-separation regression is used to remove superficial (scalp) signals.

In [ ]:
# Load fNIRS raw data
path_1 = "./data/FNIRS/DCARE_02_sub1.snirf"
path_2 = "./data/FNIRS/DCARE_02_sub2.snirf"

try:
    fnirs_data = load_fnirs(path_1, path_2, attr=None, preload=False, verbose=None)
    fnirs_participant_1 = fnirs_data[0]
    fnirs_participant_2 = fnirs_data[1]

    print("Raw fNIRS data loaded successfully.")
except Exception as e:
    print("Error loading fNIRS data:", e)

In [ ]:
def apply_short_sep_regression(raw, short_channel_names=None):
    """
    Applies short-separation regression to fNIRS data in an MNE Raw object.
    
    Parameters
    ----------
    raw : instance of mne.io.Raw
        The raw fNIRS data.
    short_channel_names : list of str, optional
        List of channel names known to be short-separation channels.
        If None, channels containing 'short' or 'ss' (case insensitive) in their name are used.
    
    Returns
    -------
    raw_reg : instance of mne.io.Raw
        A new Raw object in which the contribution of the short-separation channels
        has been regressed out from the long-separation channels.
    """
    # Make a copy so we don't alter the original
    raw_reg = raw.copy()
    data, times = raw_reg.get_data(return_times=True)  # data shape: (n_channels, n_times)
    ch_names = raw_reg.ch_names
    
    # Detect short-separation channels automatically if not provided
    if short_channel_names is None:
        short_idx = [i for i, name in enumerate(ch_names)
                     if ('short' in name.lower() or 'ss' in name.lower())]
    else:
        short_idx = [ch_names.index(name) for name in short_channel_names if name in ch_names]
    
    if len(short_idx) == 0:
        print("No short-separation channels detected; returning original raw object.")
        return raw_reg

    # Determine long-separation channel indices (channels not in short_idx)
    long_idx = [i for i in range(len(ch_names)) if i not in short_idx]
    
    # Use the average signal of short channels as the predictor
    X = np.mean(data[short_idx, :], axis=0)
    # Add a constant term to model offset
    X = np.vstack([X, np.ones_like(X)]).T  # shape (n_times, 2)
    
    # For each long channel, regress out the short-separation predictor
    new_data = data.copy()
    for i in long_idx:
        y = data[i, :]
        # Solve for beta in the model: y = beta0 * X0 + beta1 * 1
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        # Compute the fitted component
        fitted = X @ beta
        # Replace the channel data with residuals (i.e., y - fitted)
        new_data[i, :] = y - fitted
    
    # Update the Raw object's data
    raw_reg._data = new_data
    return raw_reg

In [ ]:
# Plot the original fNIRS data
fnirs_participant_1.plot(n_channels=8, duration=5, title="Raw fNIRS Data - Before Regression")

In [ ]:
# Apply short-separation regression (automatic detection of short channels)
fnirs_participant_1_clean = apply_short_sep_regression(fnirs_participant_1)

# Plot the corrected fNIRS data
fnirs_participant_1_clean.plot(n_channels=8, duration=5, title="Raw fNIRS Data - After Short-Separation Regression")

## 3. Arbitrary Methodological Choices

Preprocessing involves several choices that can affect final results. Key decisions include:
- **Filtering:**  
  The selection of lower and upper frequency cutoffs.  
  Example: A bandpass filter from 1 Hz to 40 Hz might be used for EEG.
- **Baseline Correction:**  
  Choosing the time window for baseline correction.
- **Reference Selection:**  
  For EEG, choosing a common average, linked mastoids, or another reference.
- **Epoch Rejection Criteria:**  
  Deciding on thresholds for rejecting noisy epochs.

Below is an example that demonstrates filtering and baseline correction.

In [ ]:
# For EEG, apply a bandpass filter and baseline correction to the cleaned epochs.
epo1_filtered = epo1_clean.copy().filter(l_freq=1., h_freq=40.)
epo2_filtered = epo2_clean.copy().filter(l_freq=1., h_freq=40.)

# Apply baseline correction (e.g., from -0.2 to 0 seconds)
epo1_filtered.apply_baseline((None, 0))
epo2_filtered.apply_baseline((None, 0))

print("Filtering and baseline correction applied to EEG data.")

# Visualize a few epochs after preprocessing
epo1_filtered.plot(n_epochs=5, n_channels=8, title="Preprocessed EEG Data - Participant 1")

## Discussion

- **Union vs. Intersection:**  
  The intersection approach guarantees a direct correspondence between participants, which is essential for accurate inter-brain synchrony analyses, though it may reduce the number of epochs available.

- **Artifact Rejection:**  
  ICA is a powerful tool to remove artifacts in EEG, and short-separation regression is key for improving the quality of fNIRS data. Both methods require careful parameter selection and validation.

- **Methodological Choices:**  
  Filtering, baseline correction, and reference selection are often chosen arbitrarily, but they can greatly influence the analysis outcomes. It is critical to document these choices for reproducibility.

By clearly understanding and documenting these preprocessing steps, we can ensure that the results of our hyperscanning analysis are both robust and interpretable.

## Conclusion

In this notebook, we have:
- Illustrated the difference between union and intersection of epochs.
- Demonstrated artifact rejection using ICA for EEG and short-separation regression for fNIRS.
- Discussed several arbitrary methodological choices (filtering, baseline correction, etc.) that impact the final analysis.

These preprocessing clarifications are crucial for obtaining reliable inter-brain synchrony metrics and ensuring reproducibility in hyperscanning studies.